<a href="https://colab.research.google.com/github/eeufracio/socal-comps-notebooks/blob/main/Broker_Recruiting_Scorecard(v01)ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Broker Recruiting Scorecard — CoStar Sale Comps

Turns CoStar **sale-comps** exports into a recruiting scorecard. **Universes are auto-created per CoStar
Property Type** (catch-all): Industrial, Multifamily, and Office keyword-group. Any CoStar type
containing "retail" collapses to a single **Retail** universe, and Office collapses the same way —
any subtype or parenthetical (e.g. "Office (CBD)", "Office - Medical") becomes one **Office** universe.
Blank/missing types go to **Unclassified**. Flex, Land, and every other distinct type each get their own tab. The director's touchpoint notes, broker
emails, and phone details **carry forward automatically** week to week.

**Tabs, in order:** This Week & Month to Date → Top 100 Brokers → Competitive Landscape → each
property-type universe → Review Queue → READ ME.
- **This Week & Month to Date** — previous work week on top, month-to-date below (living, auto-updates by
  run date), both across all universes.
- **Top 100 Brokers** — cumulative leaderboard, score-ranked, all universes.
- **Competitive Landscape** — firms ranked by production in the observed regions (the bar to beat).
- **Universe tabs** — one row per broker, sorted by most recent deal.

**For the director, on each broker row:**
- **Broker Email** — feeds CRM. Auto-fills: if the director types an email anywhere in the phase Notes,
  it's pulled into this column (an email typed directly in the Email cell wins).
- **Secondary Phone + Phone Type** — manual fields for the broker's mobile/secondary line; Phone Type is a
  **Mobile/Office** dropdown. Carries forward, feeds the CRM pipeline.
- **Transaction mix** — three collapsed columns (# Listings, # Buy-Side, Dominant Side). Click the **+**
  above them to expand for more insight.
- **Three touchpoint phases** in gold — Phase 1 Initial Contact (light) → Phase 2 Follow-up 1 (medium)
  → Phase 3 CRM Prelude (deep). Each has a **Method** dropdown (call/text/email/other), a **Date**, and **Notes**.

---
## ▶️ How to run it (read this first)
**Step 1 — Run everything:** menu → **Runtime → Run all**.

**Step 2 — Upload when prompted.** A file picker appears. You can select several files at once:
- **CoStar export(s)** (`.xlsx`) — one or more (e.g. per city/region). *(Required.)*
- **Last week's `broker_recruiting_ledger.csv`** — carries deal history forward. *(Skip on first run.)*
- **Last week's scorecard** (`Broker_Recruiting_Scorecard.xlsx`, the one your director marked up) —
  carries his **Status / Contacted / Notes** forward. *(Skip on first run.)*

**Step 3 — Grab your two downloads:** the new **scorecard** and the updated **master ledger**.
**Save the ledger** — you'll upload it next week.

> 💡 If a download doesn't start, click the **📁 folder icon** (left sidebar) → **/content/** →
> right-click the file → **Download**. Both files are always saved there.

---
### 🧱 CoStar layout fields
Listing **and** buyer broker: agent name, company, phone. Plus **Size, Sale Price, Sale Date,
Property Type, Comp ID, Property Address, Property City, Property State**. Step 2 prints what it found. #test synv

## ⚙️ Setup — imports & configuration
_Edit CONFIG only to change weights, week mode, or universe names._

In [10]:
import pandas as pd, numpy as np, re, difflib, datetime, io
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side, Protection
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule, CellIsRule
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.protection import SheetProtection

# ================== CONFIG ==================
WEIGHTS = {"count": 0.50, "value": 0.30, "avg": 0.20}
TOP_N = 100
WEEK_MODE = "previous_work_week"
WEEK_WINDOW_DAYS = 7
NAME_SIM_THRESHOLD = 0.86
FIRM_ALIAS = {}
OUTPUT_NAME = "Broker_Recruiting_Scorecard.xlsx"
LEDGER_NAME = "broker_recruiting_ledger.csv"
FIXED_UNIVERSES = ["Industrial", "Multifamily", "Retail", "Office"]  # always first if present
CONTACT_METHODS = "Call,Text,Email,Other"
PHONE_TYPES = "Mobile,Office"
CARRY_COLS = ["Broker Email", "Secondary Phone", "Phone Type",
              "P1 Method", "P1 Date", "P1 Notes",
              "P2 Method", "P2 Date", "P2 Notes",
              "P3 Method", "P3 Date", "P3 Notes"]
PROTECT_SHEETS = True              # lock computed cells; only contact/notes inputs stay editable
FIRM_SELF = "Foremost"             # our firm, for the Competitive Landscape benchmark
EDITABLE_INPUT_COLS = set(CARRY_COLS)
LEGACY_NOTE_MAP = {"Notes": "P1 Notes", "Outreach Notes": "P1 Notes"}  # migrate old free-form notes
# ===========================================

## 🧠 Engine — all processing, scoring, notes carry-forward, styled export
_No need to edit._

In [11]:
_norm = lambda c: re.sub(r"\s+", " ", str(c).strip().lower())
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")

def find_col(df, cands, exclude_broker=True):
    for cand in cands:
        for c in df.columns:
            cn = _norm(c)
            if cn == cand and (not exclude_broker or "broker" not in cn): return c
    return None

def firm_key(s):
    if pd.isna(s): return ""
    s = str(s).split("|")[0]; s = re.sub(r"[^a-z0-9 ]", " ", s.lower())
    s = re.sub(r"\b(commercial real estate services|commercial real estate|real estate|services|inc|llc|group|advisors|associates)\b", " ", s)
    return re.sub(r"\s+", " ", s).strip()
def canon(s): return None if pd.isna(s) else str(s).split("|")[0].strip()
def pname(f, l):
    if pd.isna(f) and pd.isna(l): return None
    return re.sub(r"\s+", " ", f"{'' if pd.isna(f) else f} {'' if pd.isna(l) else l}").strip()
def fmt_id(v):
    if pd.isna(v): return ""
    if isinstance(v, float) and v.is_integer(): return str(int(v))
    return str(v).strip()

def classify_asset(ptype):
    """Catch-all: Industrial, Multifamily, and Office keyword-group; any CoStar type containing
    'retail' collapses to 'Retail'; blank/missing → 'Unclassified'; every other distinct
    type (Flex, Land, Hospitality, ...) gets its own universe named by the full type string.
    Office collapse catches any subtype or parenthetical/suffix variant — e.g. "Office (CBD)",
    "Office - Medical", "Office/Flex" — since the match is a substring check on the full type."""
    t = re.sub(r"\s+", " ", (ptype or "").strip())
    tl = t.lower()
    if not tl: return "Unclassified"
    if re.search(r"multi\s*-?\s*family|multifamily|apartment", tl): return "Multifamily"
    if "industrial" in tl: return "Industrial"
    if "retail" in tl: return "Retail"
    if "office" in tl: return "Office"
    return t  # Flex, Land, Hospitality, Health Care, Specialty, etc. → own universe

def universe_order(present):
    extras = sorted([u for u in present if u not in FIXED_UNIVERSES])
    return [u for u in FIXED_UNIVERSES if u in present] + extras

def safe_sheet(name):
    return re.sub(r'[\\/?*\[\]:]', " ", str(name)).strip()[:31]

def detect_location(df):
    return dict(
        combined=find_col(df, ["property address", "address", "street address", "property street address"]),
        num=find_col(df, ["property street number", "street number", "street #"]),
        pre=find_col(df, ["property street pre-direction", "street pre-direction", "pre-direction"]),
        nm=find_col(df, ["property street name", "street name", "street"]),
        post=find_col(df, ["property street post-direction", "street post-direction", "post-direction"]),
        city=find_col(df, ["property city", "city"]),
        state=find_col(df, ["property state", "state"]),
        sub=find_col(df, ["submarket cluster", "submarket", "submarket code"]))

def build_credits(df):
    type_col = find_col(df, ["property type", "building use", "secondary type", "type"], exclude_broker=True)
    comp_col = find_col(df, ["comp id", "comps number", "comp number", "comps id", "comp #"])
    le_col = find_col(df, ["listing broker email", "listing broker e-mail"], exclude_broker=False)
    be_col = find_col(df, ["buyers broker email", "buyer broker email", "buyers broker e-mail"], exclude_broker=False)
    gen_email = find_col(df, ["broker email", "agent email", "email", "e-mail"], exclude_broker=False)
    loc = detect_location(df)
    def addr(r):
        if loc["combined"] and pd.notna(r.get(loc["combined"])): return str(r[loc["combined"]]).strip()
        parts = [r.get(c) for c in [loc["num"], loc["pre"], loc["nm"], loc["post"]] if c]
        parts = [str(int(p)) if isinstance(p, float) and p == int(p) else str(p) for p in parts if pd.notna(p)]
        return " ".join(parts).strip()
    df = df.copy()
    df["deal_id"] = df["PropertyID"].astype(str) + "|" + df["Sale Date"].astype(str)
    df["universe"] = df.apply(lambda r: classify_asset(r[type_col] if type_col else None), axis=1)
    rows = []
    for _, r in df.iterrows():
        a = addr(r); ct = r.get(loc["city"]) if loc["city"] else None
        stt = r.get(loc["state"]) if loc["state"] else None; sub = r.get(loc["sub"]) if loc["sub"] else None
        for role, pre in [("Listing", "Listing Broker"), ("Buyer", "Buyers Broker")]:
            nm = pname(r[f"{pre} Agent First Name"], r[f"{pre} Agent Last Name"])
            if not nm: continue
            fr = canon(r[f"{pre} Company"])
            unk = (fr is None) or (firm_key(fr) == firm_key(nm)) or (str(fr).strip().lower() == nm.lower())
            ph = r.get(f"{pre} Phone")
            esrc = (le_col if role == "Listing" else be_col) or gen_email
            ev = r.get(esrc) if esrc else None
            bem = "" if (ev is None or pd.isna(ev) or "@" not in str(ev)) else str(ev).strip()
            rows.append(dict(deal_id=r["deal_id"], comp_id=(fmt_id(r.get(comp_col)) if comp_col else ""),
                universe=r["universe"], broker=nm, firm=("(unknown)" if unk else fr),
                firm_key=("" if unk else firm_key(fr)), broker_email_src=bem,
                broker_phone=("" if pd.isna(ph) else (f"{int(ph):010d}" if str(ph).replace('.0','').isdigit() else str(ph))),
                role=role, size=r["Size"], sale_price=r["Sale Price"], close_date=r["Sale Date"],
                prop_address=a, prop_city=("" if pd.isna(ct) else ct), prop_state=("" if pd.isna(stt) else stt),
                submarket=("" if pd.isna(sub) else sub),
                prop_type=(re.sub(r"\s+", " ", str(r[type_col]).strip()) if type_col and pd.notna(r.get(type_col)) else "")))
    cr = pd.DataFrame(rows).sort_values("role").drop_duplicates(["broker", "deal_id"], keep="first")
    credited = set(cr.deal_id)
    facts = {u: dict(total_deals=g.deal_id.nunique(), brokered=g[g.deal_id.isin(credited)].deal_id.nunique())
             for u, g in df.groupby("universe")}
    have_addr = bool(loc["combined"] or loc["nm"] or loc["num"])
    return cr, facts, have_addr, {k: v for k, v in loc.items() if v}

def merge_ledger(prior, new):
    led = new.copy() if (prior is None or prior.empty) else pd.concat([prior, new], ignore_index=True)
    led["close_date"] = pd.to_datetime(led["close_date"])
    return led.drop_duplicates(["deal_id", "broker", "role"], keep="last").reset_index(drop=True)

def impute_values(led):
    led = led.copy()
    med = (led.dropna(subset=["sale_price"]).assign(p=lambda d: d.sale_price / d["size"])
              .groupby("universe")["p"].median().to_dict())
    def val(r):
        if pd.notna(r.sale_price): return pd.Series([r.sale_price, False])
        if r.universe in med: return pd.Series([r["size"] * med[r.universe], True])
        return pd.Series([np.nan, False])
    led[["value", "value_imputed"]] = led.apply(val, axis=1)
    return led, med

def _pct(s): return s.rank(pct=True) * 100 if s.nunique() > 1 else pd.Series([100.0] * len(s), index=s.index)

def score_brokers(group):
    if group.empty: return pd.DataFrame()
    out = []
    for bk, gb in group.groupby("broker"):
        d = gb.drop_duplicates("deal_id"); n = d.deal_id.nunique(); tot = d.value.sum(skipna=True)
        out.append(dict(broker=bk, firm=gb.firm.value_counts().index[0],
            phone=next((p for p in gb.broker_phone if p), ""),
            deals=n, total_value=tot, avg_value=tot / n if n else 0))
    a = pd.DataFrame(out)
    a["score"] = (WEIGHTS["count"] * _pct(a.deals) + WEIGHTS["value"] * _pct(a.total_value.fillna(0))
                  + WEIGHTS["avg"] * _pct(a.avg_value.fillna(0)))
    a["rank"] = a.score.rank(ascending=False, method="min").astype(int)
    return a.sort_values("score", ascending=False).reset_index(drop=True)

def side_counts(led_slice):
    if led_slice.empty: return {}
    vc = led_slice.drop_duplicates(["broker", "deal_id", "role"]).groupby(["broker", "role"]).size().unstack(fill_value=0)
    out = {}
    for bk, row in vc.iterrows():
        nl = int(row.get("Listing", 0)); nb = int(row.get("Buyer", 0))
        out[bk] = (nl, nb, "Balanced" if nl == nb else ("Listing" if nl > nb else "Buy-Side"))
    return out

def display_facts(led):
    out = {}
    for u, g in led.groupby("universe"):
        regs = sorted({str(x) for x in g.submarket.dropna().unique() if str(x).strip()})
        if not regs: regs = sorted({str(x) for x in g.prop_city.dropna().unique() if str(x).strip()})
        out[u] = dict(dmin=g.close_date.min(), dmax=g.close_date.max(), regions=regs,
                      deals=g.deal_id.nunique(), brokers=g.broker.nunique())
    return out

# ---------- notes / email carry-forward ----------
def extract_notes(xlsx_bytes):
    notes = {}
    try: wb = load_workbook(io.BytesIO(xlsx_bytes), data_only=True)
    except Exception: return notes
    for ws in wb.worksheets:
        hdr, cols, legacy = None, {}, {}
        for r in range(1, min(ws.max_row, 18) + 1):
            vals = {ws.cell(r, c).value: c for c in range(1, ws.max_column + 1)}
            if "Broker" in vals and (any(k in vals for k in CARRY_COLS) or any(k in vals for k in LEGACY_NOTE_MAP)):
                hdr = r
                for key in ["Broker"] + CARRY_COLS:
                    if key in vals: cols[key] = vals[key]
                for old, new in LEGACY_NOTE_MAP.items():
                    if old in vals and new not in cols: legacy[new] = vals[old]  # migrate old free-form notes
                break
        if hdr is None or "Broker" not in cols: continue
        for r in range(hdr + 1, ws.max_row + 1):
            bk = ws.cell(r, cols["Broker"]).value
            if not bk: continue
            cur = notes.get(bk, {})
            for key in CARRY_COLS:
                if key in cols:
                    v = ws.cell(r, cols[key]).value
                    if v not in (None, ""): cur[key] = v
            for new, ci in legacy.items():
                v = ws.cell(r, ci).value
                if v not in (None, "") and not cur.get(new): cur[new] = v
            if cur: notes[bk] = cur
    return notes

def _affirm(v):
    return str(v).strip().lower() in {"yes", "y", "same", "merge", "true", "x", "1", "dup", "duplicate"}

def extract_aliases(xlsx_bytes):
    """Persistent broker-merge map: reads the saved 'Broker Merges' tab + new confirmations
    in the Review Queue ('Same person?' = yes, or the exact name to keep)."""
    aliases = {}
    if not xlsx_bytes: return aliases
    try: wb = load_workbook(io.BytesIO(xlsx_bytes), data_only=True)
    except Exception: return aliases
    if "Broker Merges" in wb.sheetnames:
        ws = wb["Broker Merges"]
        for r in range(1, min(ws.max_row, 6) + 1):
            vals = {ws.cell(r, c).value: c for c in range(1, ws.max_column + 1)}
            if "Alias (merge from)" in vals and "Canonical (merge into)" in vals:
                for rr in range(r + 1, ws.max_row + 1):
                    a = ws.cell(rr, vals["Alias (merge from)"]).value
                    cc = ws.cell(rr, vals["Canonical (merge into)"]).value
                    if a and cc and str(a).strip() != str(cc).strip(): aliases[str(a).strip()] = str(cc).strip()
                break
    if "Review Queue" in wb.sheetnames:
        ws = wb["Review Queue"]; vals = {}
        for r in range(1, min(ws.max_row, 5) + 1):
            row = {ws.cell(r, c).value: c for c in range(1, ws.max_column + 1)}
            if {"Broker A", "Broker B", "Same person? (review)"} <= set(row):
                vals = row; hdr = r; break
        if vals:
            for r in range(hdr + 1, ws.max_row + 1):
                A = ws.cell(r, vals["Broker A"]).value; B = ws.cell(r, vals["Broker B"]).value
                mark = ws.cell(r, vals["Same person? (review)"]).value
                if not A or not B or mark in (None, ""): continue
                A, B, m = str(A).strip(), str(B).strip(), str(mark).strip()
                if m == A: aliases[B] = A
                elif m == B: aliases[A] = B
                elif _affirm(m): aliases[B] = A
    def resolve(n, seen=None):
        seen = seen or set()
        while n in aliases and n not in seen: seen.add(n); n = aliases[n]
        return n
    return {k: resolve(v) for k, v in aliases.items() if resolve(v) != k}

def apply_aliases(led, aliases):
    if led is None or led.empty or not aliases: return led
    led = led.copy(); led["broker"] = led["broker"].apply(lambda b: aliases.get(b, b))
    return led.drop_duplicates(["deal_id", "broker", "role"], keep="last").reset_index(drop=True)

def apply_notes(df, notes):
    if df is None or df.empty or not notes or "Broker" not in df.columns: return df
    for col in CARRY_COLS:
        if col in df.columns:
            df[col] = df.apply(lambda r: (notes.get(r["Broker"], {}).get(col, "") or r[col]), axis=1)
    return df

def resolve_email(current, broker, note_texts, source_map):
    if current and "@" in str(current): return str(current).strip()
    if source_map.get(broker): return source_map[broker]
    for t in note_texts:
        m = EMAIL_RE.search(str(t or ""))
        if m: return m.group(0)
    return current or ""

def fill_emails(df, source_map):
    if df is None or df.empty or "Broker Email" not in df.columns: return df
    ncols = [c for c in ["P1 Notes", "P2 Notes", "P3 Notes"] if c in df.columns]
    df["Broker Email"] = df.apply(lambda r: resolve_email(r["Broker Email"], r["Broker"], [r[c] for c in ncols], source_map), axis=1)
    return df

def _blank_carry(): return {c: "" for c in CARRY_COLS}

# ---------- row builders ----------
def broker_rows(led_u, scored_u, dom_type, career_deal_counts=None, career_volumes=None):
    if led_u.empty or scored_u.empty: return pd.DataFrame()
    led_u = led_u.copy(); led_u["close_date"] = pd.to_datetime(led_u.close_date)
    latest = led_u.sort_values("close_date").groupby("broker").tail(1).set_index("broker")
    sc = side_counts(led_u); out = []
    for _, b in scored_u.iterrows():
        L = latest.loc[b.broker]; nl, nb, dom = sc.get(b.broker, (0, 0, "Balanced"))
        row = {"Rank": int(b["rank"]), "Broker": b.broker, "Firm": b.firm, "Broker Phone": b.phone,
            "Broker Email": "", "Secondary Phone": "", "Phone Type": "",
            "Broker Score": round(b.score, 1), "Deals in This Universe": int(b.deals), "Volume in This Universe": b.total_value,
            "Career Total Deals": int((career_deal_counts or {}).get(b.broker, 0)), "Career Total Volume": (career_volumes or {}).get(b.broker, np.nan), "Career Dominant Type": dom_type.get(b.broker, ""),
            "# Listings": nl, "# Buy-Side": nb, "Dominant Side": dom,
            "Latest Close Date": L.close_date, "Comp ID": L.get("comp_id", ""),
            "Latest Property Address": L.prop_address, "Property City": L.prop_city,
            "Latest Sale Price": (np.nan if pd.isna(L.value) else L.value),
            "Price Basis": ("Estimated" if L.value_imputed else "Disclosed"), "Building SF": int(L["size"])}
        row.update(_blank_carry()); out.append(row)
    return pd.DataFrame(out).sort_values("Latest Close Date", ascending=False)

def activity_rows(led_window, scored_by_u, rank_map, rank_col):
    if led_window.empty: return pd.DataFrame()
    out = []
    for _, r in led_window.sort_values("close_date", ascending=False).iterrows():
        sc = scored_by_u.get(r.universe); urk = uscr = np.nan
        if sc is not None and not sc.empty and r.broker in set(sc.broker):
            s = sc[sc.broker == r.broker].iloc[0]; urk = int(s["rank"]); uscr = round(s.score, 1)
        row = {rank_col: rank_map.get(r.broker, np.nan), "Universe Rank": urk, "Universe Score": uscr,
            "Broker": r.broker, "Firm": r.firm, "Broker Phone": r.broker_phone, "Broker Email": "",
            "Secondary Phone": "", "Phone Type": "", "Side": r.role,
            "Close Date": pd.to_datetime(r.close_date), "Universe": r.universe, "Comp ID": r.get("comp_id", ""),
            "Property Address": r.prop_address, "Property City": r.prop_city,
            "Sale Price": (np.nan if pd.isna(r.value) else r.value),
            "Price Basis": ("Estimated" if r.value_imputed else "Disclosed"), "Building SF": int(r["size"])}
        row.update(_blank_carry()); out.append(row)
    return pd.DataFrame(out)

def week_window(run_date):
    d = pd.Timestamp(run_date)
    if WEEK_MODE == "trailing_days":
        return d - pd.Timedelta(days=WEEK_WINDOW_DAYS), d, f"last {WEEK_WINDOW_DAYS} days"
    monday_this = (d - pd.Timedelta(days=d.weekday())).normalize()
    start = monday_this - pd.Timedelta(days=7); end = start + pd.Timedelta(days=4)
    return start, end, "previous work week (Mon-Fri)"

def build_competitive(led):
    d = led[led.firm != "(unknown)"].drop_duplicates(["firm", "deal_id"])
    if d.empty: return pd.DataFrame()
    rows = []
    for fm, g in d.groupby("firm"):
        gv = g.drop_duplicates("deal_id"); vol = gv["value"].sum(skipna=True); n = gv.deal_id.nunique()
        regions = ", ".join(sorted({str(x) for x in g.submarket.dropna().unique() if str(x).strip()})) or "—"
        rows.append({"Firm": fm, "# Brokers": g.broker.nunique(), "Top Universe": g.universe.value_counts().index[0],
                     "Regions": regions, "Total Deals": n, "Total Volume": vol, "Avg Deal ($)": vol / n if n else 0})
    a = pd.DataFrame(rows); tv = a["Total Volume"].sum()
    a["Market Share"] = a["Total Volume"] / tv if tv else 0
    a = a.sort_values("Total Volume", ascending=False).reset_index(drop=True)
    a.insert(0, "Rank", range(1, len(a) + 1))
    return a

def month_shades(dates):
    """Alternating grey bands per calendar month (most-recent first)."""
    greys = ["FFFFFF", "EDEDED"]; out = []; last = None; idx = -1
    for d in dates:
        key = (pd.Timestamp(d).year, pd.Timestamp(d).month) if pd.notna(d) else None
        if key != last: idx += 1; last = key
        out.append(greys[idx % 2])
    return out

# ---------- styling ----------
BODY = Font(name="Arial", size=10); THIN = Side(style="thin", color="D9D9D9")
BD = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
SCORE_COLS = {"Broker Score", "Global Score", "Universe Score"}
MONEY = {"Latest Sale Price", "Sale Price", "Volume in This Universe", "Career Total Volume", "Total Value", "Avg Deal ($)", "Total Volume"}
DATEC = {"Latest Close Date", "Close Date", "Most Recent Deal"}
PCT_COLS = {"Market Share"}
DATE_INPUT_COLS = {"P1 Date", "P2 Date", "P3 Date"}
DROPDOWN_OPTIONS = {"P1 Method": CONTACT_METHODS, "P2 Method": CONTACT_METHODS, "P3 Method": CONTACT_METHODS,
                    "Phone Type": PHONE_TYPES}
METHOD_DROPDOWN_COLS = {"P1 Method", "P2 Method", "P3 Method"}
COLLAPSE_COLS = {"# Listings", "# Buy-Side", "Dominant Side"}
LIGHT_FILLS = {"FFF2CC", "FFE699", "FFD966", "F1C232"}
METHOD_COLORS = {"Call": "CFE2F3", "Text": "D9EAD3", "Email": "E4DAF0", "Other": "EFEFEF"}
COL_BODY_FILL = {"Broker Email": "FFF7E0", "Secondary Phone": "FFF7E0", "Phone Type": "FFF7E0",
    "P1 Method": "FFF7E0", "P1 Date": "FFF7E0", "P1 Notes": "FFF7E0",
    "P2 Method": "FFEFC2", "P2 Date": "FFEFC2", "P2 Notes": "FFEFC2",
    "P3 Method": "FFE49E", "P3 Date": "FFE49E", "P3 Notes": "FFE49E"}
WIDTH = {"Latest Property Address":26,"Property Address":26,"Recent Property":24,"Firm":34,"Broker":18,
         "Property City":14,"Latest Sale Price":15,"Sale Price":14,"Volume in This Universe":15,"Career Total Volume":16,"Career Total Deals":16,"Total Value":15,
         "Total Volume":15,"Broker Phone":13,"Broker Email":22,"Secondary Phone":15,"Phone Type":11,
         "Latest Close Date":15,"Close Date":12,"Most Recent Deal":15,"Primary Universe":16,"Universe":16,
         "Avg Deal ($)":14,"Comp ID":11,"Rank":6,"Weekly Rank":11,"MTD Rank":10,"Universe Rank":12,
         "Universe Score":12,"Recent Comp ID":12,"Broker Score":12,"Total Deals":10,"Deals in This Universe":16,"Global Score":11,
         "# Listings":10,"# Buy-Side":10,"Dominant Side":13,"Career Dominant Type":18,"Market Share":12,"Top Universe":16,"Regions":26,
         "# Brokers":10,"P1 Method":13,"P1 Date":13,"P1 Notes":24,"P2 Method":13,"P2 Date":13,"P2 Notes":24,
         "P3 Method":13,"P3 Date":13,"P3 Notes":24}

def disclosure(ws, lines, top=1):
    for i, (txt, bold, color) in enumerate(lines):
        c = ws.cell(top + i, 1, txt); c.font = Font(name="Arial", size=10 if bold else 9, bold=bold, italic=not bold, color=color)
    return top + len(lines) + 1

def write_grouped(ws, dfx, groups, disc_lines, top=1, do_filter=True, do_freeze=True, row_shades=None, collapse_extra=None, highlight=None):
    r0 = disclosure(ws, disc_lines, top)
    if dfx is None or dfx.empty:
        ws.cell(r0, 1, "No records for this view in the current data pull.").font = Font(name="Arial", italic=True, color="C00000")
        return r0 + 2
    flat = [c for _, _, _, cols in groups for c in cols if c in dfx.columns]
    colpos = {}; j = 1
    for label, dark, light, cols in groups:
        present = [c for c in cols if c in dfx.columns]
        if not present: continue
        light_grp = dark.upper() in LIGHT_FILLS
        ws.merge_cells(start_row=r0, start_column=j, end_row=r0, end_column=j + len(present) - 1)
        gc = ws.cell(r0, j, label); gc.fill = PatternFill("solid", fgColor=light)
        gc.font = Font(name="Arial", bold=True, size=10, color=("7F6000" if light_grp or label.startswith("RECRUIT") else "FFFFFF"))
        gc.alignment = Alignment(horizontal="center")
        for c in present:
            hc = ws.cell(r0 + 1, j, c); hc.fill = PatternFill("solid", fgColor=dark)
            hc.font = Font(name="Arial", bold=True, color=("7F6000" if light_grp else "FFFFFF"), size=9)
            hc.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True); hc.border = BD
            colpos[c] = j; j += 1
    for i, (_, rrow) in enumerate(dfx[flat].iterrows(), r0 + 2):
        di = i - (r0 + 2)
        for c in flat:
            jj = colpos[c]; v = rrow[c]; cell = ws.cell(i, jj)
            cell.value = ("" if pd.isna(v) else v); cell.font = BODY; cell.border = BD
            if c in MONEY and v != "" and pd.notna(v): cell.number_format = "$#,##0"
            elif c in PCT_COLS and v != "" and pd.notna(v): cell.number_format = "0.0%"
            elif c in DATEC and v != "" and pd.notna(v): cell.number_format = "mm/dd/yyyy"
            elif c in SCORE_COLS and v != "" and pd.notna(v): cell.number_format = "0.0"
            if c in DATE_INPUT_COLS: cell.number_format = "mm/dd/yyyy"
            if c in COL_BODY_FILL: cell.fill = PatternFill("solid", fgColor=COL_BODY_FILL[c])
            elif row_shades is not None: cell.fill = PatternFill("solid", fgColor=row_shades[di])
            if highlight is not None and highlight[di] and c not in COL_BODY_FILL:
                cell.fill = PatternFill("solid", fgColor="DDEBF7")
            if PROTECT_SHEETS and c in EDITABLE_INPUT_COLS: cell.protection = Protection(locked=False)
    last = r0 + 1 + len(dfx)
    if do_freeze: ws.freeze_panes = ws.cell(r0 + 2, 1)
    if do_filter: ws.auto_filter.ref = f"{get_column_letter(1)}{r0+1}:{get_column_letter(len(flat))}{last}"
    for c, jj in colpos.items():
        L = get_column_letter(jj); ws.column_dimensions[L].width = WIDTH.get(c, 13)
        if c in (COLLAPSE_COLS | (collapse_extra or set())):
            ws.column_dimensions[L].outline_level = 1; ws.column_dimensions[L].hidden = True
    for c in set(DROPDOWN_OPTIONS) & set(colpos):
        L = get_column_letter(colpos[c]); rng = f"{L}{r0+2}:{L}{last}"
        dv = DataValidation(type="list", formula1=f'"{DROPDOWN_OPTIONS[c]}"', allow_blank=True)
        ws.add_data_validation(dv); dv.add(rng)
        if c in METHOD_DROPDOWN_COLS:
            for val, hexc in METHOD_COLORS.items():
                ws.conditional_formatting.add(rng, CellIsRule(operator="equal", formula=[f'"{val}"'],
                                              fill=PatternFill("solid", fgColor=hexc)))
    for scol in SCORE_COLS & set(colpos):
        L = get_column_letter(colpos[scol])
        ws.conditional_formatting.add(f"{L}{r0+2}:{L}{last}",
            ColorScaleRule(start_type="min", start_color="F8696B", mid_type="percentile", mid_value=50,
                           mid_color="FFEB84", end_type="max", end_color="63BE7B"))
    if PROTECT_SHEETS:
        ws.protection = SheetProtection(sheet=True, sort=False, autoFilter=False, formatColumns=False,
            formatRows=False, selectLockedCells=False, selectUnlockedCells=False, objects=False, scenarios=False)
    return last + 2

PHASES = [("PHASE 1 — INITIAL CONTACT", "FFE699", ["P1 Method", "P1 Date", "P1 Notes"]),
          ("PHASE 2 — FOLLOW-UP 1", "FFD966", ["P2 Method", "P2 Date", "P2 Notes"]),
          ("PHASE 3 — CRM PRELUDE", "F1C232", ["P3 Method", "P3 Date", "P3 Notes"])]
_BROKER_CONTACT = ["Broker Phone","Broker Email","Secondary Phone","Phone Type"]

GROUPS_UNIV = [
    ("BROKER — RANK & PERFORMANCE", "1F3864", "2E5496",
     ["Rank","Broker","Firm"] + _BROKER_CONTACT + ["Broker Score","Deals in This Universe","Volume in This Universe","Career Total Deals","Career Total Volume","Career Dominant Type","# Listings","# Buy-Side","Dominant Side"]),
    ("LATEST DEAL — most recent at top", "375623", "548235",
     ["Latest Close Date","Latest Property Address","Property City","Latest Sale Price","Comp ID","Price Basis","Building SF"]),
] + [(lbl, fill, fill, cols) for lbl, fill, cols in PHASES]

def _groups_activity(rank_col):
    return [("RANKING", "1F3864", "2E5496", [rank_col, "Universe Rank", "Universe Score"]),
            ("BROKER INFORMATION", "1F3864", "2E5496", ["Broker","Firm"] + _BROKER_CONTACT + ["Side"]),
            ("SALE DETAILS", "375623", "548235",
             ["Close Date","Universe","Comp ID","Property Address","Property City","Sale Price","Price Basis","Building SF"]),
            ] + [(lbl, fill, fill, cols) for lbl, fill, cols in PHASES]
GROUPS_WEEK = _groups_activity("Weekly Rank")
GROUPS_MTD = _groups_activity("MTD Rank")

GROUPS_HH = [
    ("BROKER INFORMATION", "1F3864", "2E5496", ["Rank","Broker","Firm"] + _BROKER_CONTACT + ["Primary Universe"]),
    ("PERFORMANCE (all universes)", "375623", "548235",
     ["Total Deals","Total Value","Avg Deal ($)","Global Score","Most Recent Deal","Recent Comp ID","Recent Property","# Listings","# Buy-Side","Dominant Side"]),
] + [(lbl, fill, fill, cols) for lbl, fill, cols in PHASES]

GROUPS_COMP = [
    ("FIRM / BROKERAGE", "1F3864", "2E5496", ["Rank","Firm","# Brokers","Top Universe","Regions"]),
    ("PRODUCTION (in observed regions)", "375623", "548235",
     ["Total Deals","Total Volume","Avg Deal ($)","Market Share"]),
]

def disc_for(u, disp, facts, run_date, have_addr):
    d = disp.get(u, {})
    rng = "n/a" if not d else f"{pd.to_datetime(d['dmin']).date()} to {pd.to_datetime(d['dmax']).date()}"
    regs = ", ".join(d.get("regions", []) or ["(submarket field not in pull)"])
    L = [(f"{u} — Recruiting Scorecard", True, "1F3864"),
         (f"Data window (accumulated): {rng}    |    Run date: {run_date}", False, "808080"),
         (f"Regions / submarkets included: {regs}", False, "808080"),
         (f"Showing {0 if not d else d['brokers']} brokers (all sizes), sorted by MOST RECENT deal. Rank/score = secondary context.", False, "808080"),
         ("Director: fill Broker Email + Secondary Phone (Mobile/Office) for CRM. Expand the collapsed +  columns for listing/buy-side mix.", False, "7F6000"),
         ("A broker appears here if they closed \u22651 deal in this universe. 'Career Dominant Type' (collapsed +) = the property type this broker has closed the most deals in, counted across ALL universes and all time — not just this tab.", False, "808080"),
         ("'Deals in This Universe' = only deals in this asset class. A broker's Career Dominant Type may differ from this tab if they are more active in another universe.", False, "808080"),
         ("'Volume in This Universe' = $ volume closed in this asset class only. 'Career Total Volume' = $ volume across every universe.", False, "808080"),
         ("\u2195 To re-sort: click the filter arrow on any header (Latest Close Date, Rank, Broker Score) \u2192 Sort A\u2192Z / Z\u2192A.", False, "2E5496")]
    f = facts.get(u, {})
    if f: L.append((f"Most recent pull: {f['total_deals']} closed sales in this universe, {f['brokered']} with a broker.", False, "C00000"))
    if not have_addr:
        L.append(("Property address fields were not in this export — add them to the CoStar layout.", False, "C00000"))
    return L

def _kpi(ws, r, label, value, vc="000000"):
    a = ws.cell(r, 1, label); a.font = Font(name="Arial", size=10, color="333333")
    b = ws.cell(r, 2, value); b.font = Font(name="Arial", size=11, bold=True, color=vc)
    return r + 1

def build_dashboard(wb, led, wk, mtd, overall, comp, universes, new_brokers, prior_set, run_date, wlabel):
    ws = wb.create_sheet("Dashboard"); ws.sheet_properties.tabColor = "1155CC"; ws.sheet_view.showGridLines = False
    oscd = {} if overall.empty else overall.set_index("broker")["score"].to_dict()
    def vol(s): return s.drop_duplicates("deal_id")["value"].sum(skipna=True) if not s.empty else 0
    def section(r, title, color):
        ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=2)
        for cc in (1, 2):
            c = ws.cell(r, cc); c.fill = PatternFill("solid", fgColor=color)
        c = ws.cell(r, 1, title); c.font = Font(name="Arial", size=11, bold=True, color="FFFFFF")
        return r + 1
    ws.cell(1, 1, "Recruiting Dashboard").font = Font(name="Arial", size=18, bold=True, color="1F3864")
    ws.cell(2, 1, f"Run date {run_date} — headline metrics before you dive into the tabs").font = Font(name="Arial", size=9, italic=True, color="808080")
    r = 4
    r = section(r, f"THIS WEEK  ({wlabel})", "0B8043")
    r = _kpi(ws, r, "Deals closed", int(wk.deal_id.nunique()))
    r = _kpi(ws, r, "Volume", f"${vol(wk):,.0f}")
    r = _kpi(ws, r, "Brokers active", int(wk.broker.nunique()))
    twk = sorted(wk.broker.unique(), key=lambda b: -oscd.get(b, 0))[:1] if not wk.empty else []
    r = _kpi(ws, r, "Top closer this week", twk[0] if twk else "—"); r += 1
    r = section(r, "MONTH TO DATE", "E69138")
    r = _kpi(ws, r, "Deals closed", int(mtd.deal_id.nunique()))
    r = _kpi(ws, r, "Volume", f"${vol(mtd):,.0f}")
    r = _kpi(ws, r, "Brokers active", int(mtd.broker.nunique())); r += 1
    r = section(r, "PIPELINE (all-time ledger)", "3D85C6")
    r = _kpi(ws, r, "Brokers in pipeline", int(led.broker.nunique()))
    r = _kpi(ws, r, "Deals tracked", int(led.deal_id.nunique()))
    r = _kpi(ws, r, "Total volume", f"${vol(led):,.0f}")
    r = _kpi(ws, r, "New brokers this run", (len(new_brokers) if prior_set else "n/a (first run)"), vc="0B8043"); r += 1
    r = section(r, "LEADERS", "674EA7")
    if not overall.empty:
        tb = overall.iloc[0]; r = _kpi(ws, r, "Top broker (score)", f"{tb.broker}  ({tb.score:.0f})")
    if not comp.empty:
        tf = comp.iloc[0]
        r = _kpi(ws, r, "Top firm (bar to beat)", f"{tf.Firm} — {int(tf['Total Deals'])} deals / ${tf['Total Volume']:,.0f}")
        selfrow = comp[comp.Firm.astype(str).str.contains(FIRM_SELF, case=False, na=False)]
        if not selfrow.empty:
            sr = selfrow.iloc[0]; gd = int(tf['Total Deals']) - int(sr['Total Deals']); gv = tf['Total Volume'] - sr['Total Volume']
            r = _kpi(ws, r, f"{FIRM_SELF} standing", f"rank #{int(sr['Rank'])} — {int(sr['Total Deals'])} deals / ${sr['Total Volume']:,.0f}  (gap to #1: {gd} deals, ${gv:,.0f})", vc="C00000")
        else:
            r = _kpi(ws, r, f"{FIRM_SELF} standing", f"not yet in these comps — {int(tf['Total Deals'])} deals / ${tf['Total Volume']:,.0f} is the bar", vc="C00000")
    r += 1
    r = section(r, "UNIVERSES", "1F3864")
    r = _kpi(ws, r, "Count", len(universes))
    r = _kpi(ws, r, "List", ", ".join(universes))
    if prior_set and new_brokers:
        r += 1; r = section(r, "NEW BROKERS THIS RUN", "0B8043")
        ws.cell(r, 1, ", ".join(new_brokers[:40]) + ("  ..." if len(new_brokers) > 40 else "")).font = Font(name="Arial", size=10, color="0B8043")
    ws.column_dimensions["A"].width = 26; ws.column_dimensions["B"].width = 78
    if PROTECT_SHEETS:
        ws.protection = SheetProtection(sheet=True, selectLockedCells=False, selectUnlockedCells=False, objects=False, scenarios=False)

def build_merges_tab(wb, aliases):
    ws = wb.create_sheet("Broker Merges"); ws.sheet_properties.tabColor = "B7B7B7"
    ws.cell(1, 1, "Confirmed broker merges — applied automatically every run. Confirm new ones in the Review Queue; "
                  "edit/add/remove rows here to manage duplicates manually.").font = Font(name="Arial", size=9, italic=True, color="808080")
    for j, h in enumerate(["Alias (merge from)", "Canonical (merge into)"], 1):
        c = ws.cell(3, j, h); c.fill = PatternFill("solid", fgColor="1F3864"); c.font = Font(name="Arial", bold=True, color="FFFFFF", size=9)
    r = 4
    for a, cn in sorted((aliases or {}).items()):
        ws.cell(r, 1, a).font = BODY; ws.cell(r, 2, cn).font = BODY; r += 1
    ws.column_dimensions["A"].width = 28; ws.column_dimensions["B"].width = 28

def build_workbook(led, facts, have_addr, run_date, out_path, notes=None, aliases=None, prior_brokers=None):
    led = apply_aliases(led, aliases)
    led, med = impute_values(led)
    if "broker_email_src" not in led.columns: led["broker_email_src"] = ""
    source_email = (led[led.broker_email_src.astype(str).str.contains("@", na=False)]
                    .groupby("broker")["broker_email_src"].first().to_dict())
    disp = display_facts(led)
    universes = universe_order(set(led.universe.unique()))
    scored = {u: score_brokers(led[led.universe == u]) for u in universes}
    overall = score_brokers(led)
    osc = {} if overall.empty else overall.set_index("broker")["score"].to_dict()
    career_deal_counts = {} if overall.empty else overall.set_index("broker")["deals"].to_dict()
    career_volumes = {} if overall.empty else overall.set_index("broker")["total_value"].to_dict()
    rd = pd.Timestamp(run_date)
    cd = pd.to_datetime(led.close_date).dt.normalize()
    wstart, wend, wlabel = week_window(run_date)
    wk = led[(cd >= wstart) & (cd <= wend)]
    mstart = rd.replace(day=1).normalize()
    mtd = led[(cd >= mstart) & (cd <= rd.normalize())]
    comp = build_competitive(led)
    prior_set = {(aliases or {}).get(b, b) for b in (prior_brokers or set())}
    new_brokers = sorted(set(led.broker) - prior_set) if prior_set else []

    wb = Workbook(); wb.remove(wb.active)

    # 0) DASHBOARD (headline KPIs)
    build_dashboard(wb, led, wk, mtd, overall, comp, universes, new_brokers, prior_set, run_date, wlabel)

    # 1) THIS WEEK & MONTH TO DATE  (two stacked sections, all universes)
    wrank = {b: i + 1 for i, b in enumerate(sorted(wk.broker.unique(), key=lambda b: -osc.get(b, 0)))}
    mrank = {b: i + 1 for i, b in enumerate(sorted(mtd.broker.unique(), key=lambda b: -osc.get(b, 0)))}
    wk_df = fill_emails(apply_notes(activity_rows(wk, scored, wrank, "Weekly Rank"), notes), source_email)
    mtd_df = fill_emails(apply_notes(activity_rows(mtd, scored, mrank, "MTD Rank"), notes), source_email)
    ws = wb.create_sheet("This Week & Month to Date"); ws.sheet_properties.tabColor = "0B8043"
    wkL = [("This Week's Hitters — previous work week", True, "1F3864"),
           (f"Deals closed {wstart.date()} to {wend.date()} ({wlabel})  |  Run date {run_date}  |  all universes", False, "808080"),
           (f"{wk.deal_id.nunique()} deal(s), {wk.broker.nunique()} broker(s). Weekly Rank = top closers this week.", False, "808080")]
    nxt = write_grouped(ws, wk_df, GROUPS_WEEK, wkL, top=1, do_filter=False, do_freeze=False)
    mtdL = [("Month to Date — living activity (auto-updates by run date)", True, "1F3864"),
            (f"All deals closed {mstart.date()} to {rd.date()} (month of {rd.strftime('%B %Y')}), most recent on top, all universes.", False, "808080"),
            (f"{mtd.deal_id.nunique()} deal(s), {mtd.broker.nunique()} broker(s) month-to-date.", False, "808080"),
            ("\u2195 Filter/sort enabled on this section (Close Date, MTD Rank, etc.).", False, "2E5496")]
    write_grouped(ws, mtd_df, GROUPS_MTD, mtdL, top=nxt + 1, do_filter=True, do_freeze=False)

    # 2) TOP 100 BROKERS (all universes, score-ranked)
    allb = overall.copy()
    if not allb.empty:
        prim = led.groupby("broker").apply(lambda g: g.universe.value_counts().index[0]).to_dict()
        recent = led.sort_values("close_date").groupby("broker").tail(1).set_index("broker")
        scd = side_counts(led)
        allb["Primary Universe"] = allb.broker.map(prim)
        allb["# Listings"] = allb.broker.map(lambda b: scd.get(b, (0,0,""))[0])
        allb["# Buy-Side"] = allb.broker.map(lambda b: scd.get(b, (0,0,""))[1])
        allb["Dominant Side"] = allb.broker.map(lambda b: scd.get(b, (0,0,"Balanced"))[2])
        allb["Most Recent Deal"] = allb.broker.map(pd.to_datetime(recent.close_date))
        allb["Recent Comp ID"] = allb.broker.map(recent["comp_id"] if "comp_id" in recent else pd.Series(dtype=object))
        allb["Recent Property"] = allb.broker.map(recent.prop_address.replace("", np.nan))
        hh = allb.sort_values("score", ascending=False).head(TOP_N).copy()
        hh = hh.rename(columns={"rank":"Rank","broker":"Broker","firm":"Firm","phone":"Broker Phone",
                                "deals":"Total Deals","total_value":"Total Value","avg_value":"Avg Deal ($)","score":"Global Score"})
        hh["Global Score"] = hh["Global Score"].round(1)
        for c in CARRY_COLS:
            if c not in hh.columns: hh[c] = ""
        hh = fill_emails(apply_notes(hh, notes), source_email)
    else: hh = pd.DataFrame()
    ws = wb.create_sheet("Top 100 Brokers"); ws.sheet_properties.tabColor = "E69138"
    hhL = [("Top 100 Brokers — cumulative leaderboard", True, "1F3864"),
           (f"Top {TOP_N} brokers across ALL universes and all cities/dates accumulated so far. Run date {run_date}.", False, "808080"),
           ("Living list: each run unions new CoStar pulls into the ledger and re-ranks everyone. Sorted by score.", False, "808080"),
           ("\u2195 To re-sort: filter arrow on any header (Global Score, Total Deals, Most Recent Deal) \u2192 Sort A\u2192Z / Z\u2192A.", False, "2E5496")]
    write_grouped(ws, hh, GROUPS_HH, hhL)

    # 3) PROPERTY-TYPE UNIVERSES (dynamic, in preferred order; month-shaded, date-sorted)
    if "prop_type" not in led.columns: led["prop_type"] = ""
    _pt = led.assign(_disp=lambda d: d.apply(
        lambda r: r["prop_type"] if r["prop_type"] else r["universe"], axis=1))
    dom_type = (_pt.drop_duplicates(["broker", "deal_id"]).groupby("broker")["_disp"]
                .agg(lambda s: s.value_counts().index[0]).to_dict())
    for u in universes:
        ws = wb.create_sheet(safe_sheet(u.replace(" / ", " ")))
        ws.sheet_properties.tabColor = "3D85C6"
        rows = fill_emails(apply_notes(broker_rows(led[led.universe == u], scored[u], dom_type, career_deal_counts, career_volumes), notes), source_email)
        shades = month_shades(rows["Latest Close Date"].tolist()) if not rows.empty else None
        write_grouped(ws, rows, GROUPS_UNIV, disc_for(u, disp, facts, run_date, have_addr),
                      row_shades=shades, collapse_extra={"Comp ID", "Price Basis", "Building SF", "Career Total Deals", "Career Total Volume", "Career Dominant Type"})

    # 4) REVIEW QUEUE
    names = sorted(led.broker.unique()); pq = []
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            rt = difflib.SequenceMatcher(None, names[i].lower(), names[j].lower()).ratio()
            if rt >= NAME_SIM_THRESHOLD or names[i].split()[-1] == names[j].split()[-1]:
                pq.append({"Broker A":names[i],"Firm A":led[led.broker==names[i]].firm.iloc[0],
                           "Broker B":names[j],"Firm B":led[led.broker==names[j]].firm.iloc[0],
                           "Name Similarity":round(rt,2),"Same person? (review)":""})
    rv = wb.create_sheet("Review Queue"); rv.sheet_properties.tabColor = "999999"; rvdf = pd.DataFrame(pq)
    if rvdf.empty: rv.cell(1,1,"No possible same-person matches.").font = Font(name="Arial", italic=True)
    else:
        rv.cell(1,1,"Mark 'Same person?' = yes (or type the name to keep) to merge them next run. Confirmed merges move to the Broker Merges tab.").font = Font(name="Arial", size=9, italic=True, color="808080")
        for j,c in enumerate(rvdf.columns,1):
            hc=rv.cell(2,j,c); hc.fill=PatternFill("solid",fgColor="1F3864"); hc.font=Font(name="Arial",bold=True,color="FFFFFF",size=9)
        confirm_col = list(rvdf.columns).index("Same person? (review)") + 1
        for i,(_,row) in enumerate(rvdf.iterrows(),3):
            for j,c in enumerate(rvdf.columns,1):
                cell = rv.cell(i,j,row[c]); cell.font=BODY
                if PROTECT_SHEETS and j == confirm_col: cell.protection = Protection(locked=False)
        rv.column_dimensions["B"].width=34; rv.column_dimensions["D"].width=34
        if PROTECT_SHEETS:
            rv.protection = SheetProtection(sheet=True, selectLockedCells=False, selectUnlockedCells=False, objects=False, scenarios=False)

    # 5) BROKER MERGES (persistent alias store, editable)
    build_merges_tab(wb, aliases)

    # 6) COMPETITIVE LANDSCAPE (firms by production in observed regions)
    comp = build_competitive(led)
    ws = wb.create_sheet("Competitive Landscape"); ws.sheet_properties.tabColor = "674EA7"
    if comp.empty:
        cL = [("Competitive Landscape", True, "1F3864"), ("No firm production data yet.", False, "808080")]
        write_grouped(ws, comp, GROUPS_COMP, cL)
    else:
        top = comp.iloc[0]
        regions_all = ", ".join(sorted({r for s in comp.Regions for r in str(s).split(", ") if r and r != "—"})) or "the observed market"
        selfrow = comp[comp.Firm.astype(str).str.contains(FIRM_SELF, case=False, na=False)]
        cL = [("Competitive Landscape — firm production in the observed regions", True, "1F3864"),
              (f"Run date {run_date}  |  {len(comp)} firms  |  Regions: {regions_all}", False, "808080"),
              (f"Bar to beat: {top.Firm} leads with {int(top['Total Deals'])} deals / ${top['Total Volume']:,.0f} "
               f"({top['Market Share']*100:.0f}% share) across {int(top['# Brokers'])} broker(s).", False, "C00000")]
        if not selfrow.empty:
            sr = selfrow.iloc[0]; gd = int(top['Total Deals']) - int(sr['Total Deals']); gv = top['Total Volume'] - sr['Total Volume']
            cL.append((f"{FIRM_SELF} (highlighted): rank #{int(sr['Rank'])} of {len(comp)} — {int(sr['Total Deals'])} deals / "
                       f"${sr['Total Volume']:,.0f} ({sr['Market Share']*100:.0f}% share). Gap to #1: {gd} deals, ${gv:,.0f}.", False, "1155CC"))
        else:
            cL.append((f"{FIRM_SELF} is not yet in these comps — match {int(top['Total Deals'])} deals / ${top['Total Volume']:,.0f} to lead.", False, "1155CC"))
        cL.append(("\u2195 Sort by Total Volume, Total Deals, or Market Share via the filter arrows.", False, "2E5496"))
        hl = comp.Firm.astype(str).str.contains(FIRM_SELF, case=False, na=False).tolist()
        write_grouped(ws, comp, GROUPS_COMP, cL, highlight=hl)

    # 6) READ ME (last)
    rm = wb.create_sheet("READ ME"); rm.sheet_properties.tabColor = "434343"; rm.sheet_view.showGridLines = False
    nnotes = 0 if not notes else len(notes)
    ulist = "  |  ".join(universes)
    L = [("Broker Recruiting Scorecard", 16, True, "1F3864"),
         (f"Generated {run_date}  |  Living workbook — re-run weekly with the ledger + last scorecard.", 9, False, "808080"), ("",10,0,"000000"),
         ("Tab order", 12, True, "1F3864"),
         ("  Dashboard → This Week & Month to Date → Top 100 Brokers → universes → Review Queue → Broker Merges → Competitive Landscape → READ ME", 10,0,"000000"), ("",10,0,"000000"),
         ("Dashboard", 12, True, "1155CC"),
         ("  One-screen headline KPIs (this week, MTD, pipeline, leaders, new brokers) before you open the detail tabs.", 10,0,"000000"), ("",10,0,"000000"),
         ("Duplicate brokers (Review Queue + Broker Merges)", 12, True, "1F3864"),
         ("  Review Queue flags likely same-person pairs. Mark 'Same person?' = yes (or type the name to keep) and next run they merge.", 10,0,"000000"),
         ("  Confirmed merges live on the Broker Merges tab and re-apply every run, so a broker's history stays unified.", 10,0,"000000"), ("",10,0,"000000"),
         ("Locked cells", 12, True, "1F3864"),
         ("  Computed cells are protected; only contact + the 3 note phases are editable. In Google Sheets you'll get a warning if you edit a locked cell.", 10,0,"000000"),
         ("  To unlock everything: Data → Protected sheets and ranges → remove (or set PROTECT_SHEETS=False in the notebook).", 10,0,"000000"), ("",10,0,"000000"),
         ("Universes — auto-created per CoStar Property Type (catch-all)", 12, True, "1F3864"),
         (f"  This run: {ulist}", 10,0,"000000"),
         ("  Industrial, Multifamily, and Office keyword-group (any Office subtype or parenthetical, e.g. 'Office (CBD)', collapses into one Office universe). Any CoStar type containing 'retail' collapses to 'Retail'; blank/missing → 'Unclassified'. Flex, Land, and every other distinct type each get their own tab.", 10,0,"000000"), ("",10,0,"000000"),
         ("Scoring (composite 0-100): Deal count 50% / Total value 30% / Avg deal 20%", 10,0,"000000"),
         ("  Universe tabs sort by most recent deal; Top 100 sorts by score.", 10,0,"000000"), ("",10,0,"000000"),
         ("Broker contact (feeds CRM pipeline)", 12, True, "7F6000"),
         ("  Broker Email auto-fills from notes if typed there. Add Secondary Phone + Phone Type (Mobile/Office dropdown).", 10,0,"000000"),
         ("  3 gold touchpoint phases — each: Method (Call/Text/Email/Other dropdown, color-coded), Date, Notes. Carried forward.", 10,0,"000000"),
         (f"  ({nnotes} broker contact/note record(s) carried into this run.)", 10,0,"000000"), ("",10,0,"000000"),
         ("This Week & Month to Date", 12, True, "1F3864"),
         ("  Top section = previous work week. Lower section = month-to-date (auto-updates by run date), newest on top.", 10,0,"000000"), ("",10,0,"000000"),
         ("Competitive Landscape", 12, True, "1F3864"),
         ("  Firms ranked by production in the observed regions (deals, volume, market share) — the bar to beat.", 10,0,"000000"), ("",10,0,"000000"),
         ("Sorting (Google Sheets too): filter arrow on any header → Sort A→Z / Z→A. If absent: Data → Create a filter.", 10, True, "2E5496"), ("",10,0,"000000"),
         ("Tab colors: green = This Week & MTD, orange = Top 100, blue = universes, purple = Competitive Landscape, grey = Review Queue/READ ME.", 9, False, "808080"),
         ("Collapsed (+) columns: universe tabs hide Comp ID / Price Basis / Building SF and the transaction-mix + Career Dominant Type — click + to expand.", 9, False, "808080")]
    for i,(t,sz,b,col) in enumerate(L,1): rm.cell(i,1,t).font = Font(name="Arial", size=sz, bold=bool(b), color=col)
    rm.column_dimensions["A"].width = 112

    wb.save(out_path)
    return scored, med, led


## 1️⃣ Upload your file(s)
**Run this cell, then use the file picker.** Select — in any combination:
- your **CoStar export(s)** `.xlsx`  *(required)*
- last week's **`broker_recruiting_ledger.csv`**  *(carries deal history; skip on first run)*
- last week's **marked-up scorecard** `.xlsx`  *(carries the director's notes; skip on first run)*

The notebook figures out which file is which automatically.

In [12]:
from google.colab import files

print("="*72)
print("UPLOAD NOW — pick any of these (the file picker is opening below):")
print("  • CoStar export(s) .xlsx          (required — one or several)")
print("  • broker_recruiting_ledger.csv    (weekly run: carries deal history)")
print("  • last week's scorecard .xlsx     (weekly run: carries the director's notes)")
print("="*72)
up = files.upload()

export_frames, prior_ledger, prior_scorecard_bytes, export_names = [], None, None, []
for fn, content in up.items():
    low = fn.lower()
    if low.endswith(".csv") and "ledger" in low:
        prior_ledger = pd.read_csv(io.BytesIO(content)); print(f"  ✓ Ledger loaded: {fn}")
    elif low.endswith((".xlsx", ".xls")):
        try: sheets = pd.ExcelFile(io.BytesIO(content)).sheet_names
        except Exception: sheets = []
        if "Heavy Hitters Log" in sheets or "READ ME" in sheets:
            prior_scorecard_bytes = content; print(f"  ✓ Prior scorecard loaded — notes will carry forward: {fn}")
        else:
            export_frames.append(pd.read_excel(io.BytesIO(content))); export_names.append(fn); print(f"  ✓ Export loaded: {fn}")
    elif low.endswith(".csv"):
        export_frames.append(pd.read_csv(io.BytesIO(content))); export_names.append(fn); print(f"  ✓ Export loaded (csv): {fn}")

if not export_frames:
    raise ValueError("No CoStar export found — re-run this cell and select at least one comps .xlsx.")

export_df = pd.concat(export_frames, ignore_index=True, sort=False)
print(f"\n{len(export_frames)} export(s) combined → {export_df.shape[0]} rows.")
print("Prior ledger:   ", "NONE — fresh start." if prior_ledger is None else f"{len(prior_ledger)} rows (history carries forward).")
print("Prior scorecard:", "NONE — notes start blank." if prior_scorecard_bytes is None else "loaded (notes carry forward).")

UPLOAD NOW — pick any of these (the file picker is opening below):
  • CoStar export(s) .xlsx          (required — one or several)
  • broker_recruiting_ledger.csv    (weekly run: carries deal history)
  • last week's scorecard .xlsx     (weekly run: carries the director's notes)


Saving CostarExport (36).xlsx to CostarExport (36).xlsx
Saving CostarExport (35).xlsx to CostarExport (35).xlsx
Saving CostarExport (34).xlsx to CostarExport (34).xlsx
Saving CostarExport (33).xlsx to CostarExport (33).xlsx
Saving CostarExport (32).xlsx to CostarExport (32).xlsx
  ✓ Export loaded: CostarExport (36).xlsx
  ✓ Export loaded: CostarExport (35).xlsx
  ✓ Export loaded: CostarExport (34).xlsx
  ✓ Export loaded: CostarExport (33).xlsx
  ✓ Export loaded: CostarExport (32).xlsx

5 export(s) combined → 1036 rows.
Prior ledger:    NONE — fresh start.
Prior scorecard: NONE — notes start blank.


## 2️⃣ Build the scorecard
**Just run this cell.** It carries notes forward, scores everyone, writes the workbook, and reports
broker counts and which fields it detected.

In [13]:
RUN_DATE = datetime.date.today().isoformat()

notes   = extract_notes(prior_scorecard_bytes) if prior_scorecard_bytes else None
aliases = extract_aliases(prior_scorecard_bytes) if prior_scorecard_bytes else None
prior_brokers = set(prior_ledger["broker"]) if prior_ledger is not None and "broker" in prior_ledger.columns else None

# --- Diagnosing and fixing 'Sale Date' KeyError ---
print("Current columns in export_df:", export_df.columns.tolist())
# If 'Sale Date' is not in the list above, but you see a column like 'SaleDate' or 'Date of Sale',
# uncomment the line below and replace 'Your Actual Date Column Name' with the correct column name.
# export_df = export_df.rename(columns={'Your Actual Date Column Name': 'Sale Date'})
# -------------------------------------------------

credits, facts, have_addr, detected = build_credits(export_df)
ledger = merge_ledger(prior_ledger, credits)
out_xlsx   = "/content/" + OUTPUT_NAME
scored, med, ledger = build_workbook(ledger, facts, have_addr, RUN_DATE, out_xlsx,
                                     notes=notes, aliases=aliases, prior_brokers=prior_brokers)
out_ledger = "/content/" + LEDGER_NAME
ledger.to_csv(out_ledger, index=False)

print(f"Run date {RUN_DATE}  |  master ledger now holds {len(ledger)} broker-deal rows.")
print(f"Notes carried forward for {0 if not notes else len(notes)} broker(s).")
print(f"Broker merges applied: {0 if not aliases else len(aliases)}.\n")
print("Brokers ranked per universe (all deal sizes):")
for u, s in scored.items():
    print(f"   • {u}: {0 if s.empty else len(s)}")
print("\nProperty/ID fields detected:", detected if detected else "NONE")
if not have_addr:
    print("   ⚠ No property ADDRESS field found — add 'Property Address' (or street number+name)")
    print("     plus 'Property City' / 'Property State' to the CoStar layout.")

Current columns in export_df: ['Comp ID', 'PropertyID', 'Property Address', 'Property City', 'Property Zip Code', 'Property State', 'Property Type', 'Secondary Type', 'Listing Broker Agent First Name', 'Listing Broker Agent Last Name', 'Listing Broker Phone', 'Listing Broker Company', 'Listing Broker City', 'Listing Broker State', 'Buyers Broker Agent First Name', 'Buyers Broker Agent Last Name', 'Buyers Broker Phone', 'Buyers Broker Company', 'Buyers Broker City', 'Buyers Broker State', 'Size', 'Price Per SF (Net)', 'Submarket Cluster', 'Submarket Code', 'Sale Condition', 'Multi-Sale Name', 'Portfolio Name', 'Transaction Notes', 'Sale Price', 'Sale Date']


/tmp/ipykernel_17215/1942958372.py:576: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  prim = led.groupby("broker").apply(lambda g: g.universe.value_counts().index[0]).to_dict()


Run date 2026-06-30  |  master ledger now holds 559 broker-deal rows.
Notes carried forward for 0 broker(s).
Broker merges applied: 0.

Brokers ranked per universe (all deal sizes):
   • Industrial: 129
   • Multifamily: 62
   • Retail: 114
   • Flex: 11
   • Office: 75
   • Office (Community Center): 1
   • Office (Neighborhood Center): 1

Property/ID fields detected: {'combined': 'Property Address', 'city': 'Property City', 'state': 'Property State', 'sub': 'Submarket Cluster'}


## 3️⃣ Download your two files
**Run this cell.** Keep BOTH: the scorecard (open it) and the master ledger (upload it next week).
If a download doesn't auto-start, use the **📁 Files panel** → `/content/` → right-click → Download.

In [14]:
import time

print("="*72)
print("✅ DONE — downloading TWO files. Keep BOTH:")
print(f"   1. {OUTPUT_NAME}   ← the scorecard (your director marks this up)")
print(f"   2. {LEDGER_NAME}   ← the MASTER LEDGER (save it; upload it next week)")
print("="*72)

for path in [out_xlsx, out_ledger]:
    try:
        files.download(path); time.sleep(2)
    except Exception:
        print(f"   (auto-download skipped for {path})")

print("\nIf either file did NOT download automatically:")
print("   • Click the 📁 folder icon on the LEFT sidebar → open /content/")
print(f"   • Right-click {OUTPUT_NAME} and {LEDGER_NAME} → Download")
print("\nNEXT WEEK: upload the new export(s) + this ledger + the scorecard (after the director marks it up).")

✅ DONE — downloading TWO files. Keep BOTH:
   1. Broker_Recruiting_Scorecard.xlsx   ← the scorecard (your director marks this up)
   2. broker_recruiting_ledger.csv   ← the MASTER LEDGER (save it; upload it next week)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


If either file did NOT download automatically:
   • Click the 📁 folder icon on the LEFT sidebar → open /content/
   • Right-click Broker_Recruiting_Scorecard.xlsx and broker_recruiting_ledger.csv → Download

NEXT WEEK: upload the new export(s) + this ledger + the scorecard (after the director marks it up).


## 📈 The living workbook — weekly cadence & scaling
- **Each week:** export fresh comps → run this notebook → upload the new export(s) **+** last week's
  ledger **+** the director's marked-up scorecard. History and notes both carry forward.
- **Add a city/region:** include its export in the upload. Deals de-duplicate on PropertyID + close
  date + broker + side, so overlaps are safe.
- **Universes:** Industrial and Multifamily keyword-group; any CoStar type containing "retail" → **Retail**;
  blank/missing → **Unclassified**; Flex, Office, Land, and every other distinct type get their own tab. All deal sizes.
- **Tuning (CONFIG cell):** `WEIGHTS` (frequency-led), `WEEK_MODE`, `HEAVY_HITTERS_TOP_N`, `UNIVERSES`.
- **Keep `broker_recruiting_ledger.csv` safe — it's your cumulative history.**